<a href="https://colab.research.google.com/github/guitorte/audio/blob/main/music-to-midi/notebooks/MIDI_Engine_Bakeoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎼 Bake-off de engines música → MIDI

Roda várias engines no **mesmo trecho** de uma música e compara velocidade, métricas de qualidade, piano rolls e o som de cada MIDI.

| Etapa | Engine | O que é | Roda em |
|---|---|---|---|
| Separação | **htdemucs_6s** | o que o `song_to_midi` usa hoje | GPU (torch) |
| Separação | **BS-RoFormer SW** | RoFormer de 6 stems, piano e baixo bem mais limpos | GPU (torch) |
| Stems com altura | **Basic Pitch** | o que o `song_to_midi` usa hoje, mesmos presets | CPU (ONNX) |
| Voz | **GAME** (OpenVPI, v1.0.3) | canto → notas, pesos ONNX oficiais | GPU/CPU (ONNX) |
| Bateria | **ADTOF** | o que o `song_to_midi` usa hoje | GPU (torch) |
| Bateria | **ADT_STR** (2026) | transformer de bateria, com velocity | GPU, ambiente Python 3.12 próprio |
| Mix inteiro | **MuScriptor** (Kyutai × Mirelo) | música inteira → MIDI multi-track, com tempo e compassos | GPU (torch) |

**Antes de começar**
1. *Runtime ▸ Change runtime type ▸ **T4 GPU***.
2. **Só para o MuScriptor:** os pesos são restritos. Entre no Hugging Face, abra [MuScriptor/muscriptor-medium](https://huggingface.co/MuScriptor/muscriptor-medium) (e *-small*/*-large* se for usar), aceite a licença (CC BY-NC 4.0, uso não comercial), crie um token de leitura e adicione no painel 🔑 **Secrets** do Colab como `HF_TOKEN`, com acesso do notebook ligado. Sem isso o MuScriptor só é pulado.
3. Rode as células na ordem. As células 3 a 10 podem ser re-rodadas com outra música ou outro trecho.

Sem um MIDI de referência não há gabarito: as colunas de qualidade são **aproximações** (veja as notas no fim). Os piano rolls e a célula de escuta são o teste de verdade.

Os resultados vão para `<pasta de músicas>/midi-bakeoff/<música>__<início>s+<duração>s/`.

In [ ]:
#@title 1 · Instalar (≈ 3–5 min)
import importlib.metadata as _md, sys as _sys
_antes = {m: getattr(_sys.modules.get(m), "__version__", None) for m in ("numpy", "torch")}

# audio-separator: os dois separadores (traz também o onnxruntime-gpu, usado pelo Basic Pitch e pelo GAME).
# Ele importa audioread sem declarar a dependência.
!pip -q install "audio-separator[gpu]>=0.48" audioread
# MuScriptor (código MIT; os pesos são restritos, veja o topo do notebook).
!pip -q install "muscriptor>=0.4"
# Basic Pitch sem TensorFlow (sem wheels para este Python): cai no modelo ONNX.
# resampy >= 0.4.3 porque as versões antigas importam pkg_resources, que o setuptools novo removeu.
!pip -q install --no-deps basic-pitch==0.4.0
!pip -q install "resampy>=0.4.3" pretty_midi mir_eval pandas
# ADTOF-pytorch só existe no GitHub.
!pip -q install git+https://github.com/xavriley/ADTOF-pytorch.git
# FluidSynth + soundfont General MIDI, só para a prévia soar melhor.
!apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null

_mudou = [m for m, v in _antes.items() if v and v != _md.version(m)]
if _mudou:
    print(f"⚠️ {', '.join(_mudou)} mudou de versão já carregado: Runtime ▸ Restart session e siga da célula 2.")
else:
    print("✓ instalado — não precisa reiniciar")

In [ ]:
#@title 2 · Conectar o Drive e listar as músicas
DRIVE_FOLDER = "áudio"  #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

REPO, BRANCH = "/content/audio", "main"
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                       "https://github.com/guitorte/audio.git", REPO]).returncode:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/guitorte/audio.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/music-to-midi")
from modules import bakeoff as B
U = B.U

if B.device() != "cuda":
    print("⚠️ Sem GPU: Runtime ▸ Change runtime type ▸ T4 GPU. Em CPU demora muito.\n")
FOLDER = U.find_folder(DRIVE_FOLDER)
FILES = U.list_audio(FOLDER)
print(f"📁 {FOLDER}  ({len(FILES)} músicas)")
for i, f in enumerate(FILES, 1):
    print(f"{i:3d}. {f.name}")

In [ ]:
#@title 3 · Escolher a música, o trecho e as engines
SONG = "1"  #@param {type:"string"}
#@markdown Número da lista acima ou parte do nome.
EXCERPT_START = 30  #@param {type:"number"}
EXCERPT_SECONDS = 60  #@param {type:"number"}
#@markdown Todas as engines recebem o mesmo trecho. `EXCERPT_SECONDS = 0` usa o resto da música (lento no MuScriptor e no ADT_STR).
#@markdown ---
#@markdown **Separação**
HTDEMUCS_6S = True  #@param {type:"boolean"}
BS_ROFORMER_SW = True  #@param {type:"boolean"}
#@markdown **Transcrição por stem**
BASIC_PITCH = True  #@param {type:"boolean"}
ADTOF = True  #@param {type:"boolean"}
GAME_SIZE = "medium"  #@param ["off", "small", "medium", "large"]
GAME_LANGUAGE = ""  #@param ["", "en", "ja", "zh", "yue"]
#@markdown Vazio = neutro (use para português).
ADT_STR = True  #@param {type:"boolean"}
#@markdown **Transcrição do mix inteiro** (precisa do segredo `HF_TOKEN`, veja o topo)
MUSCRIPTOR_SIZE = "medium"  #@param ["off", "small", "medium", "large"]
#@markdown ---
REFERENCE_MIDI = ""  #@param {type:"string"}
#@markdown Opcional: um MIDI que bate exatamente com a música (projeto da DAW, export MIDI do Suno…), caminho completo ou nome de um `.mid` na pasta. Dá a acurácia real (F1 de notas) de cada engine. O tempo 0 dele tem que ser o tempo 0 da música.

chosen = U.select_files(FILES, SONG)
if len(chosen) != 1:
    print(f"⚠️ '{SONG}' casou com {len(chosen)} músicas; usando a primeira.")
SONG_PATH = chosen[0]
REF = None
if REFERENCE_MIDI:
    REF = Path(REFERENCE_MIDI) if os.path.isabs(REFERENCE_MIDI) else FOLDER / REFERENCE_MIDI
    assert REF.exists(), f"MIDI de referência não encontrado: {REF}"
    REF = str(REF)

MIX = B.start(SONG_PATH, EXCERPT_START, EXCERPT_SECONDS)
LENGTH = B.audio_seconds(MIX)
assert LENGTH > 1, "O trecho ficou vazio: EXCERPT_START passou do fim da música?"
print(f"🎵 {SONG_PATH.name}: trecho de {LENGTH:.1f}s a partir de {EXCERPT_START}s")
from IPython.display import Audio, display
display(Audio(str(MIX)))

In [ ]:
#@title 4 · Separar (o BS-RoFormer SW baixa ≈ 670 MB na primeira vez)
names = [n for n, on in (("htdemucs_6s", HTDEMUCS_6S), ("bs_roformer_sw", BS_ROFORMER_SW)) if on]
B.run_separations(MIX, names)
for sep, stems in B.STEM_FILES.items():
    print(sep, "->", ", ".join(sorted(stems)))

In [ ]:
#@title 5 · Transcrever os stems (Basic Pitch, ADTOF, GAME, ADT_STR)
B.run_stem_engines(
    MIX,
    basic_pitch_on=BASIC_PITCH,
    adtof_on=ADTOF,
    game_size=None if GAME_SIZE == "off" else GAME_SIZE,
    game_language=GAME_LANGUAGE,
    adt_str_on=ADT_STR,
)

In [ ]:
#@title 6 · MuScriptor no mix inteiro
if MUSCRIPTOR_SIZE == "off":
    print("MuScriptor desligado.")
else:
    B.ensure_hf_token()
    B.run_muscriptor(MIX, MUSCRIPTOR_SIZE)

In [ ]:
#@title 7 · Resultados
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 70)
SIZE = None if MUSCRIPTOR_SIZE == "off" else MUSCRIPTOR_SIZE

SPEED = B.runs_table()
QUALITY = B.quality_table(REF, EXCERPT_START, LENGTH)
SYSTEMS = B.systems(SIZE)
SYSTEMS_T = B.systems_table(SYSTEMS, MIX, REF, EXCERPT_START, LENGTH)

display(Markdown("### Velocidade  \n`x_realtime` = segundos de áudio por segundo de trabalho (maior é mais rápido). "
                 "Na primeira vez, `seconds` inclui o download dos modelos; `load_seconds` aparece quando a engine expõe. "
                 "`peak_vram_gb` só cobre as engines em torch."))
display(SPEED)
display(Markdown("### Por stem  \n`chroma_vs_stem`: concordância de harmonia com o stem separado (0–1, maior é melhor). "
                 "`drum_onset_f1`: concordância com os ataques detectados no stem de bateria. "
                 "`stem_rms_db`: volume do stem; um stem muito abaixo dos outros (digamos, abaixo de −50 dBFS) é quase só vazamento, "
                 "então as notas escritas para ele são inventadas. "
                 "`ref_note_f1`: acurácia real contra o seu MIDI de referência, se houver."))
display(QUALITY.sort_values(["category", "candidate"]).reset_index(drop=True))
display(Markdown("### Música inteira  \nAs faixas com altura de cada sistema, renderizadas e comparadas com o mix original."))
display(SYSTEMS_T)

In [ ]:
#@title 8 · Piano rolls por stem
import copy
import matplotlib.pyplot as plt
ref_by_cat = B.split_by_category(B.load_midi(REF)) if REF else {}
for cat in B.CATEGORIES:
    cands = {label: B.load_midi(p) for label, p in B.CANDIDATES[cat].items()}
    if not cands:
        continue
    if cat in ref_by_cat:
        r = copy.deepcopy(ref_by_cat[cat])
        r.adjust_times([EXCERPT_START, EXCERPT_START + LENGTH], [0, LENGTH])
        cands = {"REFERÊNCIA": r, **cands}
    B.piano_roll_grid(cands, cat, LENGTH)
    plt.show()

In [ ]:
#@title 9 · Ouvir
HEAR = "sistemas"  #@param ["sistemas", "vocals", "drums", "bass", "guitar", "piano", "other"]
#@markdown Escolha outra opção e rode a célula de novo.
from IPython.display import Audio, display, Markdown
SF2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"

def play(title, y, sr):
    display(Markdown(f"**{title}**"))
    display(Audio(y, rate=sr))

if HEAR == "sistemas":
    display(Markdown("**Trecho original**")); display(Audio(str(MIX)))
    for name, pm in SYSTEMS.items():
        play(name, *B.render_preview(pm, LENGTH, SF2))
else:
    stem, sep = B.reference_stem(HEAR)
    if stem:
        display(Markdown(f"**stem {HEAR} ({sep})**")); display(Audio(str(stem)))
    for label, p in B.CANDIDATES[HEAR].items():
        play(label, *B.render_preview(B.load_midi(p), LENGTH, SF2))

In [ ]:
#@title 10 · Salvar os MIDIs e as tabelas no Drive
OVERWRITE = False  #@param {type:"boolean"}
dst = FOLDER / "midi-bakeoff" / B.RUN_DIR.name
if B.save_outputs(dst, SYSTEMS, {"speed": SPEED, "per_stem": QUALITY, "systems": SYSTEMS_T}, OVERWRITE):
    for f in sorted(dst.glob("*.mid")):
        print(" ", f.name)

## Como ler os resultados

* **Velocidade.** Separação e transcrição são cronometradas à parte. Os stems de um trecho ficam guardados em `/content/bakeoff_work/runs/`, então re-rodar a célula 4 com o mesmo trecho não separa de novo.
* **`chroma_vs_stem` / `chroma_vs_mix`** renderizam o MIDI com senoides e comparam o *croma* (quais das 12 classes de nota soam, quadro a quadro) com o áudio. Premia a harmonia certa na hora certa e pune trechos perdidos ou inventados, mas não enxerga erro de oitava nem separa duas notas do mesmo acorde. Um estudo sobre métricas de transcrição viu que a distância de croma acompanha bem a preferência de ouvintes ([arXiv 2608.04511](https://arxiv.org/abs/2608.04511)). Diferenças de poucos centésimos são empate.
* **O croma por stem é medido contra o stem do BS-RoFormer SW** quando ele existe, para o vazamento de um separador não favorecer a própria transcrição.
* **`drum_onset_*`** compara as batidas do MIDI com os ataques que o librosa detecta no stem de bateria. Não diz *qual* peça foi tocada.
* **`ref_note_f1`** é o único número de acurácia real: início da nota a até 50 ms e altura certa, o padrão MIREX/mir_eval. Precisa do `REFERENCE_MIDI`. Uma música sintética (por exemplo do Suno, com o export MIDI) é o jeito mais fácil de ter um.
* **MuScriptor** não usa stems: as faixas dele são encaixadas num stem pelo programa General MIDI. Ele não escreve velocity (todas as notas no mesmo volume) e os pesos são para uso não comercial.
* **GAME** sorteia as fronteiras das notas (D3PM), então duas rodadas podem sair um pouco diferentes. Os pesos dele também parecem ser não comerciais (CC BY-NC-SA 4.0, segundo um espelho; não consegui abrir a página oficial).
* **ADT_STR** roda num ambiente Python 3.12 próprio (ele fixa o torch 2.8), criado no primeiro uso em `/content/bakeoff_work/envs/adt_str`.
* Depois de um `git push` com correções, rode *Runtime ▸ Disconnect and delete runtime* antes de usar de novo, senão o Colab continua com o código antigo em `/content/audio`.